# Chapter 11 &mdash; The Missing Basis Case

**Concept 5 of the Chapter 11 decomposition:** *The Missing Basis Case: How CFG Rules Populate a Language*

`S -> (S) | SS` denotes $\emptyset$ &mdash; without a terminal-only right-hand side, nothing is ever generated.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Missing-Basis-Case/Concept-Missing-Basis-Case.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A grammar is an **inductive definition**, and an inductive definition with no **basis
case** defines nothing.

`S -> (S) | SS` has two recursive rules and no way to stop. Every sentential form
still contains an `S`, so **no sentence is ever produced** and $L(G)=\emptyset$.

The fix is one production: `S -> ''` (or `S -> ()`).

The general rule: **some right-hand side must be terminal-only**, and more precisely
every nonterminal you actually use must be **productive** &mdash; able to derive *some*
terminal string. A nonterminal that cannot is dead weight, and if $S$ is one, the
language is empty.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### The broken grammar and the fixed one

In [ ]:
Broken = mkg({'S': ["(S)", "SS"]})
Fixed  = mkg({'S': ["", "(S)", "SS"]})

### Productivity: which nonterminals derive anything at all?

In [ ]:
def productive(G):
    # least fixed point: A is productive if some RHS is made of terminals
    # and already-productive nonterminals
    P = set()
    changed = True
    while changed:
        changed = False
        for A, rhss in G['P'].items():
            if A in P: continue
            if any(all(x in G['Sigma'] or x in P for x in r) for r in rhss):
                P.add(A); changed = True
    return P

## 3. Tests

The broken grammar generates **nothing**.

In [ ]:
print("L(Broken) up to length 10 :", language(Broken, 10))
assert language(Broken, 10) == []
print("L(Fixed)  up to length 6  :", language(Fixed, 6))
assert language(Fixed, 6)

Productivity analysis says why.

In [ ]:
print("productive nonterminals, Broken :", productive(Broken))
print("productive nonterminals, Fixed  :", productive(Fixed))
assert productive(Broken) == set()
assert productive(Fixed) == {'S'}
print("\nS is unproductive in Broken, so L = {} -- and no amount of")
print("rewriting will ever change that.")

A grammar can be *partly* broken: one dead nonterminal among several.

In [ ]:
Partly = mkg({'S': ["aA", "b"], 'A': ["aA"]})
print("L(Partly) :", language(Partly, 5))
print("productive :", sorted(productive(Partly)))
assert 'A' not in productive(Partly) and 'S' in productive(Partly)
assert set(language(Partly, 5)) == {'b'}
print("\nS survives only through 'S -> b'; every string through A dies.")

Either basis case works.

In [ ]:
for basis in ["", "()"]:
    G = mkg({'S': [basis, "(S)", "SS"]})
    print("basis %-4r -> L up to 4 : %s" % (basis, language(G, 4)))
    assert language(G, 4)

The check is cheap; run it on every grammar you write.

In [ ]:
for name, G in [('Broken', Broken), ('Fixed', Fixed), ('Partly', Partly)]:
    ok = G['S'] in productive(G)
    print("%-8s start symbol productive? %s" % (name, ok))

## 4. Exercises


1. Write a grammar where $S$ is productive but the language is still empty. (Hint: reachability.)
2. What is the analogue of an unproductive nonterminal in a recursive function?
3. Add a productivity check to `mkg` that warns you.

In [ ]:
# Your work for the exercises above.